In [2]:
import numpy as np
import pandas as pd 

from astroquery.sdss import SDSS 
from astropy.table import Table
from astropy.io import fits
import numpy as np
from tqdm import tqdm
import time

import random

In [1]:
from requests.exceptions import ConnectionError, Timeout, ReadTimeout
from http.client import RemoteDisconnected
from urllib3.exceptions import ProtocolError

In [3]:
SDSS.TIMEOUT = 300

In [ ]:
class DataFetcher:

    '''
    Fetch spectral data from the remote sql server 
    '''

    def __init__(self, spectral_class, output_directory = "/kaggle/working"):
        self.spectral_class = spectral_class
        self.output_directory = output_directory

    def get_table(self):
        if self.spectral_class == "F":
            num = 4290
        else:
            num = 4285

        # running the sql query to get the fiberid, mjd, plate for the stars
        sql_query = f'''SELECT TOP {num}
            specobjid, ra, dec,plate, mjd, fiberid, subclass, z, elodieTeff, elodieLogG, elodieFeH
            FROM SpecObj
            WHERE class = 'STAR' AND zWarning = 0 
              AND subclass LIKE '{self.spectral_class}%'
              AND (snMedian_u + snMedian_g + snMedian_r + snMedian_i + snMedian_z)/5.0 > 10
              AND elodieTeff IS NOT NULL AND elodieLogG IS NOT NULL AND elodieFeH IS NOT NULL'''

        res = SDSS.query_sql(sql_query, timeout = 600)

        return res

    @classmethod
    def retrieve_spectra(cls, output_dir, plate_id, mjd, fiber_id, failed_ids, chunk_num, n, spectral_class):

        save_dir = os.path.join(output_dir, f"flux_array_{spectral_class}", "spectrum_fits", f"Chunk_{chunk_num}")
        os.makedirs(save_dir, exist_ok=True)
        
        spectra = None 
        
        for attempt in range(5):
            try:
                SDSS.TIMEOUT = 300  # Increase timeout to 5 min
                
                spectra = SDSS.get_spectra(
                    plate=plate_id, mjd=mjd, fiberID=fiber_id, data_release=19
                        )
                if spectra is not None:

                    data_file = spectra[0]
                    primary_hdu = data_file[0]
                    secondary_hdu = data_file[1]
    
                    header = primary_hdu.header
    
                    try:
                        obj_id = header["THING_ID"]    # unique object identifier
                    except:
                        obj_id = header["spec_id"]
    
                    secondary_data = secondary_hdu.data
                    flux = [row[0] for row in secondary_data]
                    flux[0] = obj_id
                    
                    #save block
                    save_path = os.path.join(save_dir, f"spectrum_{n}.npy")
                    np.save(save_path, flux)
                    
                    break  # success
    
            except (ConnectionError, Timeout, ReadTimeout, socket.timeout, 
                                RemoteDisconnected, ProtocolError, TimeoutError) as e:
                
                wait_time = 10 * (2 ** attempt) + random.uniform(0, 5)
                print(f"{type(e).__name__}: Retry {attempt+1}/5 after {wait_time:.1f}s "
                      f"for plate={plate_id}, mjd={mjd}, fiber={fiber_id}")
                time.sleep(wait_time)
    
        if spectra is None:
            failed_ids.append((plate_id, mjd, fiber_id)) 
        

    def chunk_wise_loader(self, chunk_size,output_directory):
        res = self.get_table()
        print(len(res))

        flux_array = []
        spectral_class = self.spectral_class
        
        n = len(res)//chunk_size
        count = 0
        for i in tqdm(range(n)):
            try:
                new_res = res[chunk_size * i : chunk_size * (i + 1)]
            except:
                new_res = res[chunk_size * i :]
            spectra_list = []
            failed_ids = []
            
            for j in range(chunk_size):
                plate_id, mjd, fiber_id = new_res[j]["plate"], new_res[j]["mjd"], new_res[j]["fiberid"]
                
                DataFetcher.retrieve_spectra(output_directory, plate_id, mjd, fiber_id, failed_ids,chunk_num = i, n = j, spectral_class = spectral_class)

                # checker block

    def consolidate_data(self):
        '''
        Consolidating all the npy spectra flux arrays
        '''
        input_directory = "/kaggle/input/sdss-dissertation-data-1"
        directory_path = f"{input_directory}/flux_array_{self.spectral_class}/spectrum_fits"
        
        chunk_list_dir = os.listdir(directory_path)
        spectra_list = []
        
        for chunk_dir in tqdm(chunk_list_dir):
            chunk_spectra_list_dir = os.listdir(f"{directory_path}/{chunk_dir}")
        
            for j in chunk_spectra_list_dir:
                spectra = np.load(f"{directory_path}/{chunk_dir}/{j}")
                spectra_list.append(spectra)

        np.save(f"{self.output_directory}/spectra_{self.spectral_class}", np.array(spectra_list, dtype = object))


In [22]:
spectra_class = DataFetcher("F")

spectra_class.chunk_wise_loader(chunk_size = 500,output_directory = "/home/arbiter/projects/Survey-invariant-generalization/data")

4290


  0%|          | 0/8 [00:00<?, ?it/s]WARNING: NoResultsWarning: Query returned no results. [astroquery.sdss.core]


Failed to fetch plate=3106, mjd=54738, fiberID=16, skipping.


  0%|          | 0/8 [11:23<?, ?it/s]


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))